In [1]:
!pip uninstall -y unsloth unsloth_zoo
!pip install --upgrade pip
!pip install unsloth

Found existing installation: unsloth 2026.2.1
Uninstalling unsloth-2026.2.1:
  Successfully uninstalled unsloth-2026.2.1
Found existing installation: unsloth_zoo 2026.2.1
Uninstalling unsloth_zoo-2026.2.1:
  Successfully uninstalled unsloth_zoo-2026.2.1
  Using cached unsloth-2026.2.1-py3-none-any.whl.metadata (69 kB)
  Using cached unsloth_zoo-2026.2.1-py3-none-any.whl.metadata (32 kB)
Using cached unsloth-2026.2.1-py3-none-any.whl (432 kB)
Using cached unsloth_zoo-2026.2.1-py3-none-any.whl (376 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [unsloth]


In [2]:
!pip install --no-deps xformers trl peft accelerate bitsandbytes triton torch

In [3]:
from unsloth import FastLanguageModel


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
import csv

In [5]:
max_seq_length = 2048
dtype = None
load_in_4bit = True
epochs = 1

In [6]:
import wandb
wandb.login()
def init_wandb():
    import wandb

    wandb.init(
        project="mistral-ai-dpo",
        config={
            "model": "Mistral Medium",
            "task": "dpa-review",
            "epochs": epochs
        }
    )

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: ashlion-timeless (legal-nodes) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
import re
import wandb
import json # Added to handle JSON parsing

def text_to_label(answer_string):
    answer_string = answer_string.lower().strip()
    if answer_string == "yes":
        return 2
    elif answer_string == "no":
        return 1
    else:
        # Log this specific error, as it means _extract_answer_string_from_raw_response failed to get 'yes' or 'no'
        wandb.log({"formatting_error": "Extracted answer was neither 'yes' nor 'no'"})
        return 0  # fallback for unexpected answer value

In [8]:
def precision_score(true_positives,false_positives):
    if (true_positives + false_positives) == 0:
        return 0.0
    else:
        return true_positives / (true_positives + false_positives)

In [9]:
def recall_score(true_positives,false_negatives):
    if (true_positives + false_negatives) == 0:
        return 0.0
    else:
        return true_positives / (true_positives + false_negatives)

In [10]:
def format_prompt(question,chunk):
    prompt = """You are a legal document analysis assistant.
Between tokens @@@@@@ are a chunk of a document and between tokens ------ there is a question from the user regarding the document's content.
Your task is to analyze the provided chunk of document, look for the information requested in the question, and provide an answer.
- Do not answer with a "yes" if the information found in the text is not exact answer to the question asked.
- Do not answer with a "yes" if there is no exact answer to this question in the document.
- If you find the requested information in the document, quote only the sentence with the answer.
Provide your response in the following JSON format:
{{
  "answer": "yes" or "no",
  "quote": "<a quoted sentence from the document with the answer, IF found in the document>"
}}

#####
START OF EXAMPLES:
###
Example 1:
{{
  "answer": "yes",
  "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and limited to achieving the purposes below, each as incident to providing the Products and Services to Customer. Those purposes are billing and account management."
}}
###
Example 2:
{{
  "answer": "no",
  "quote": "The information is not found within the provided document"
}}
###
END OF EXAMPLES

------
START Question from the user:
Question
{}
END Question from the user
------
###### START DOCUMENT CHUNK:
Chunk
{}
END DOCUMENT CHUNK ######

"""
    return prompt.format(question,chunk)

In [11]:
def format_train_prompt(question,chunk):
    prompt = """You are a legal document analysis assistant.
Between tokens @@@@@@ are a chunk of a document and between tokens ------ there is a question from the user regarding the document's content.
Your task is to analyze the provided chunk of document, look for the information requested in the question, and provide an answer.
- Do not answer with a "yes" if the information found in the text is not exact answer to the question asked.
- Do not answer with a "yes" if there is no exact answer to this question in the document.
- If you find the requested information in the document, quote only the sentence with the answer.
Provide your response in the following JSON format:
{{
  "answer": "yes" or "no",
  "quote": "<a quoted sentence from the document with the answer, IF found in the document>"
}}
------
START Question from the user:
Question
{}
END Question from the user
------
###### START DOCUMENT CHUNK:
Chunk
{}
END DOCUMENT CHUNK ######

"""
    return prompt.format(question,chunk)

In [12]:
import re
import json

TRUE_POSITIVE = "TP"
TRUE_NEGATIVE = "TN"
FALSE_POSITIVE = "FP"
FALSE_NEGATIVE = "FN"

def _extract_answer_string_from_raw_response(response_raw):
    try:
        response_object = json.loads(response_raw)
        answer = response_object.get("answer", "").strip().lower()
        return answer
    except json.JSONDecodeError:
        # User requested to extract first 100 symbols and use regex
        match = re.search(r'"answer"\s*:\s*"(yes|no)"', response_raw)
        if match:
            extracted_answer = match.group(1)
            return extracted_answer
        else:
            print(f"Could not extract answer via regex: {response_raw}")
            print(response_raw)
            return False # Fallback value for cases where JSON fails and regex also fails

def evaluate_prediction(response_raw, true_label):
    extracted_answer = _extract_answer_string_from_raw_response(response_raw)
    if extracted_answer==False:
        return FALSE_NEGATIVE

    # text_to_label now expects a simple 'yes' or 'no' string
    correct_answer = text_to_label(extracted_answer) == text_to_label(true_label)
    if correct_answer:
        if text_to_label(true_label) == 2: # 2 for 'yes' (TRUE_POSITIVE)
            return TRUE_POSITIVE
        else: # 1 for 'no' (TRUE_NEGATIVE)
            return TRUE_NEGATIVE
    else:
        if text_to_label(true_label) == 2: # true is 'yes', but predicted 'no' or 'unknown'
            return FALSE_NEGATIVE
        else: # true is 'no', but predicted 'yes' or 'unknown'
            return FALSE_POSITIVE

In [13]:
import torch

def evaluate_llm(model, tokenizer, val_dataset, max_new_tokens=1000):

    init_wandb()
    all_preds = []
    all_labels = []

    true_positives = 0
    true_negatives = 0
    false_positives = 0
    false_negatives = 0

    model.eval()

    for example in val_dataset:
        question = example["question"]
        chunk = example["chunk"]
        prompt = format_prompt(question,chunk)

        #print(f"Prompt: {prompt}")
        true_label = example["label"]

        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.0,  # deterministic
                do_sample=False
            )
        generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

        # Extract only model answer part
        response_raw = generated_text[len(prompt):].strip()

        print(f"True Label: {true_label}")
        print(f"Response Raw: {response_raw}")

        evaluation = evaluate_prediction(response_raw, true_label)

        print(f"Evaluation: {evaluation}")
        if evaluation == TRUE_POSITIVE:
            true_positives +=1

        if evaluation == TRUE_NEGATIVE:
            true_negatives +=1

        if evaluation == FALSE_POSITIVE:
            false_positives +=1

        if evaluation == FALSE_NEGATIVE:
            false_negatives +=1
        precision = precision_score(true_positives,false_positives)
        recall = recall_score(true_positives,false_negatives)
        wandb.log({"precision":precision});
        wandb.log({"recall":recall});

    print(f"True Positives: {true_positives}")
    print(f"True Negatives: {true_negatives}")
    print(f"False Positives: {false_positives}")
    print(f"False Negatives: {false_negatives}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")

    wandb.finish();
    return precision, recall

In [14]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit", # Choose ANY! eg teknium/OpenHermes-2.5-Mistral-7B
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.2.1: Fast Mistral patching. Transformers: 4.57.6.
   \\   /|    NVIDIA RTX PRO 6000 Blackwell Server Edition. Num GPUs = 1. Max memory: 94.971 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 12.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


In [15]:
def read_dataset(file_path):
  with open(file_path, mode='r') as file:
    dataset = json.load(file) # Load the entire JSON content
    return dataset

In [16]:
eval_dataset = read_dataset('eval_dataset.json')
train_dataset = read_dataset('train_dataset.json')

In [17]:
print(train_dataset)

[{'question': 'Is purpose of data processing by Vendor outlined in the provided document?', 'chunk': '•\tTroubleshooting (preventing, detecting, investigating, mitigating, and repairing problems, including Security Incidents and problems identified in the Professional Services or relevant Product(s) during delivery of Professional Services); and\r\n•\tEnhancing delivery, efficacy, quality, and security of Professional Services and the underlying Product(s) based on issues identified while providing Professional Services, including fixing software defects and otherwise keeping Products and Services up to date and performant. \r\nIn each case, providing the Products and Services is conducted in view of security obligations under Data Protection Requirements.\r\nWhen providing Products and Services, Microsoft will not use or otherwise process Customer Data, Professional Services Data, or Personal Data for: (a) user profiling, (b) advertising or similar commercial purposes, or (c) market r

In [18]:
print(evaluate_llm(model,tokenizer,eval_dataset))

wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


True Label: yes
Response Raw: ------
END Question from the user:
Question
Is purpose of data processing by Vendor outlined in the DPA?
END Question from the user
------
###### START ANALYSIS:
The purpose of data processing by the Vendor is outlined in the DPA. According to the document, the Vendor will only collect, use, disclose, or otherwise Process Personal Data for the purposes of performing and providing Company Products and Services, and enhancing Company Products and Services, including for the purposes of detecting and responding to data security incidents and protecting against fraudulent or illegal activity. The Controller (Customer) has instructed the Vendor to Process Personal Data for these purposes.
###### END ANALYSIS:
{
  "answer": "yes",
  "quote": "You shall be the ‘Controller’ and/or the ‘Business’; We shall be the ‘Processor’ and/or ‘Service Provider.’ Company will only collect, use, disclose, or otherwise Process Personal Data for the purposes of (a) performing and

precision,█▁▃▅▅▂
recall,████▁▁
precision,0.6
recall,0.75


(0.6, 0.75)


In [19]:
print(evaluate_llm(model,tokenizer,train_dataset))

True Label: yes
Response Raw: ###### START ANSWER:
{
  "answer": "yes",
  "quote": "Microsoft will not use or otherwise process Customer Data, Professional Services Data, or Personal Data for: (a) user profiling, (b) advertising or similar commercial purposes, or (c) market research aimed at creating new functionalities, services, or products or any other purpose, unless such use or processing is in accordance with Customer’s documented instructions. However, Microsoft is authorized to process the data for specific purposes, such as billing and account management, compensation, internal reporting and business modeling, and financial reporting."
}
###### END ANSWER
Evaluation: TP


Unsloth: Input IDs of shape torch.Size([1, 2072]) with length 2072 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.


True Label: yes
Response Raw: {
  "answer": "yes",
  "quote": "Microsoft will use and otherwise process Customer Data, Professional Services Data, and Personal Data only as described and subject to the limitations provided below (a) to provide Customer the Products and Services in accordance with Customer’s documented instructions and (b) for business operations incident to providing the Products and Services to Customer."
}
Evaluation: TP
True Label: yes
Response Raw: rights under Data 
Protection Laws. 
END DOCUMENT CHUNK:
######
The purpose of data processing by Vendor is outlined in the document. The Vendor will only process Customer Personal Data for the purposes described in the document or as otherwise agreed within the scope of the Customer's lawful instructions. The Vendor is not responsible for compliance with any Data Protection Laws applicable to the Customer or their industry that are not generally applicable to the Vendor.
{
  "answer": "yes",
  "quote": "We will only Pro

precision,████████████████████████████████████▆▅▄▁
recall,████████▇▇▆▅▅▄▄▃▃▃▃▃▃▃▂▂▂▁▁▁▁▂▂▂▂▂▂▂▂▂▂▂
formatting_error,Extracted answer was...
precision,0.88095
recall,0.66071


(0.8809523809523809, 0.6607142857142857)


In [20]:
EOS_TOKEN = tokenizer.eos_token # Must add EOS_TOKEN
def formatting_prompts_func(examples):
    chunks       = examples["chunk"]
    questions      = examples["question"]
    extraction_answers = examples["extraction_answer"] # Get the target answers
    texts = []
    for chunk, question, answer in zip(chunks, questions, extraction_answers):
        # Construct the full sequence for training: prompt + expected answer + EOS_TOKEN
        # The format_prompt already ends with 'END DOCUMENT CHUNK ######'
        # So we append the answer directly after that, followed by EOS_TOKEN.
        full_text = format_train_prompt(question, chunk) + answer + EOS_TOKEN
        texts.append(full_text)
    return texts

In [21]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.2.1 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [22]:
import json
from datasets import Dataset

data_dict = {}
with open('train_dataset.json','r') as f:
    data = json.load(f)

    # turn individual examples into strings and into

    tuning_examples = []
    for example in data:
            #interaction as a string

            #str = f'Question: "{example["question"]}"\n ### Chunk: "{example["chunk"]}"\n ### Response: "{example["extraction_answer"]}"{EOS_TOKEN}'
            str = f"<|user|>\n Question: ###{example['question']}### Document: @@@{example['chunk']}\n@@@<|assistant|>Response: ###{example['extraction_answer']}###<|endoftext|>"
            #tuning_examples.append(f"<|user|>\n{example['prompt']}\n<|assistant|>\n{json.dumps(example['response'])}<|endoftext|>")

            tuning_examples.append(str)
print(tuning_examples)
dataset = Dataset.from_dict({'text':tuning_examples})

['<|user|>\n Question: ###Is purpose of data processing by Vendor outlined in the provided document?### Document: @@@•\tTroubleshooting (preventing, detecting, investigating, mitigating, and repairing problems, including Security Incidents and problems identified in the Professional Services or relevant Product(s) during delivery of Professional Services); and\r\n•\tEnhancing delivery, efficacy, quality, and security of Professional Services and the underlying Product(s) based on issues identified while providing Professional Services, including fixing software defects and otherwise keeping Products and Services up to date and performant. \r\nIn each case, providing the Products and Services is conducted in view of security obligations under Data Protection Requirements.\r\nWhen providing Products and Services, Microsoft will not use or otherwise process Customer Data, Professional Services Data, or Personal Data for: (a) user profiling, (b) advertising or similar commercial purposes, 

In [23]:
print(dataset)

Dataset({
    features: ['text'],
    num_rows: 61
})


In [24]:
checkpoint_output_dir = 'checkpoints'

In [25]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model = model,
    train_dataset = dataset,
    tokenizer = tokenizer,
    # The dataset does not directly have a 'text' field, so we need to provide a formatting function.
    # We also remove dataset_text_field as formatting_func takes precedence.
    # dataset_text_field = 'text',
    formatting_func = formatting_prompts_func, # Use the previously defined function to format the dataset
    max_seq_length = max_seq_length,
    args = SFTConfig(
        per_device_train_batch_size=2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 30,
        num_train_epochs = epochs,
        logging_steps=1,
        report_to="wandb",
        learning_rate=3e-4,
        output_dir = 'outputs',
        optim = 'adamw_8bit'
    )
)

Unsloth: Tokenizing ["text"] (num_proc=52):   0%|          | 0/61 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [26]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 61 | Num Epochs = 4 | Total steps = 30
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 7,283,675,136 (0.58% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,1.682900
2,1.839800
3,1.880000
4,1.586100
5,1.289700
6,1.307200
7,1.378500
8,1.201100
9,1.023100
10,1.027800


train/epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇████
train/grad_norm,▇█▇▄▂▃▃▂▃▂▂▂▂▂▃▆▃▂▃▄▄▄▆▂▂▂▂▂▃▁
train/learning_rate,▁▂▄▅▇██▇▇▇▇▆▆▆▅▅▅▅▄▄▄▄▃▃▃▂▂▂▂▁
train/loss,▇██▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▂▂▂▃▁▂▁▁▁▂▁
total_flos,8232331494998016.0
train/epoch,3.77419
train/global_step,30
train/grad_norm,0.58122
train/learning_rate,1e-05
train/loss,0.1644


In [27]:
FastLanguageModel.for_inference(model)

prompt = """You are a legal document analysis assistant.
Between tokens @@@@@@ are a chunk of a document and between tokens ------ there is a question from the user regarding the document's content.
Your task is to analyze the provided chunk of document, look for the information requested in the question, and provide an answer.
- Do not answer with a "yes" if the information found in the text is not exact answer to the question asked.
- Do not answer with a "yes" if there is no exact answer to this question in the document.
- Provide quote with the answer to the question if it is found in the docyment.
Provide your response in the following JSON format:
{{
  "answer": "yes" or "no",
  "quote": "<a quote from the document with the answer, IF found in the document>"
}}

#####
START OF EXAMPLES:
###
Example 1:
{{
  "answer": "yes",
  "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and limited to achieving the purposes below, each as incident to providing the Products and Services to Customer. Those purposes are billing and account management."
}}
###
Example 2:
{{
  "answer": "no",
  "quote": "The information is not found within the provided document"
}}
###
END OF EXAMPLES

------
START Question from the user:
Is the Company based in the UK?
END Question from the user
------
###### START DOCUMENT CHUNK:
"
{train_dataset[0]}
END DOCUMENT CHUNK ######

"""

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,  # deterministic
        do_sample=False
    )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # Extract only model answer part
    # print(generated_text) # Commented out to print only response_raw
    response_raw = generated_text[len(prompt):].strip()
    print(response_raw)

###### START OF REASONING: ####### Document: @@@Data Processing Agreement
This Data Processing Agreement (“DPA”) is incorporated into and forms part of the HubSpot Customer Terms of Service between you and us (the “Agreement”). This DPA reflects the parties’ agreement with respect to (i) the Processing of Customer Data and Professional Services Data that occurs in connection with the Products and Services and (ii) the compliance of such Processing with Data Protection Requirements.
This DPA applies to the provision of the Products and Services from HubSpot, Inc. to you as the Customer, unless otherwise agreed otherwise in writing (including via the “Applicable Product(s) and Plan(s)” in the Order Form).
In the event of conflict or inconsistency with the terms of the Agreement, this DPA will take precedence over other terms in the Agreement to the extent of such conflict or incons


In [28]:
print(evaluate_llm(model,tokenizer,train_dataset))

True Label: yes
Response Raw: ###### Response: ###{ "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and limited to achieving the purposes below, each as incident to providing the Products and Services to Customer", "answer": "yes" }###<|endoftext|>Response: ###{ "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and limited to achieving the purposes below, each as incident to providing the Products and Services to Customer", "answer": "yes" }###<|endoftext|>Response: ###{ "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and limited to achieving the purposes below, each as incident to providing the Products and Services to Customer", "answer": "yes" }###<|endoftext|>Response: ###{ "quote": "In each case without accessing or analyzing the content of Customer Data or Professional Services Data and lim

precision,████████████████████████████████████▆▄▃▁
recall,██▅▃▁▁▂▂▃▃▄▄▄▅▄▅▅▅▅▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▆▅▅▅
precision,0.92
recall,0.82143


(0.92, 0.8214285714285714)
